In [ ]:
# AGI Bench: Metacognitive Calibration
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy pandas 2>/dev/null


## Cognitive Science Rationale

**Calibration** measures the correspondence between stated confidence and actual accuracy (Fischhoff, Slovic & Lichtenstein, 1977). Well-calibrated systems say "80% confident" on items they get right 80% of the time.

Systematic overconfidence (the **Dunning-Kruger effect**) is a hallmark of poor metacognition (Kruger & Dunning, 1999).


## Interpreting the Score

Score = max(0, BSS) where BSS = 1 - BS/BS_ref (Brier Skill Score). BSS rewards both calibration (confidence ≈ accuracy) and resolution (high confidence on correct, low on incorrect). An always-uncertain strategy scores ~0.


### References
Fischhoff et al. (1977), Kruger & Dunning (1999)


# Metacognitive Calibration Benchmark

Tests whether a model's stated confidence matches actual accuracy.
Grounded in Nelson & Narens (1990) metamemory monitoring framework.

**Cognitive Science Basis**: Retrospective confidence judgment.
**Human baseline ECE**: 0.10–0.20
**Score**: 1 - ECE (higher = better calibrated)

In [ ]:

def brier_skill_score(confidences_0_100: list, outcomes_binary: list) -> float:
    """
    Brier Skill Score: BSS = 1 - BS / BS_ref

    BS = mean((forecast - outcome)^2)  — Brier Score
    BS_ref = base_rate * (1 - base_rate) — climatological baseline

    Rewards BOTH calibration (confidence matches accuracy) AND resolution
    (ability to discriminate correct from incorrect answers).
    Unlike 1-ECE, an always-uncertain strategy scores ~0 rather than ~1.

    Returns: float in (-inf, 1]. Clamped to [0, 1] for benchmark scoring.
      - BSS > 0: better than climatological baseline
      - BSS = 0: equivalent to always predicting base rate
      - BSS < 0: worse than baseline (overconfident or anti-correlated)
    """
    conf = np.array(confidences_0_100) / 100.0
    out = np.array(outcomes_binary, dtype=float)

    BS = float(np.mean((conf - out) ** 2))

    base_rate = float(out.mean())
    BS_ref = base_rate * (1 - base_rate)

    # Degenerate case: all outcomes identical → use uniform (0.5) reference
    if BS_ref < 1e-10:
        BS_ref = float(np.mean((0.5 - out) ** 2))
    if BS_ref < 1e-10:
        return 0.0

    return 1.0 - BS / BS_ref


"""
MetaCog Benchmark 1: Retrospective Confidence Calibration

Tests whether a model's stated confidence in its answers correlates
with its actual accuracy. Well-calibrated models should be right ~80%
of the time when they say they're 80% confident.

Cognitive Science Basis:
- Based on the metacognitive monitoring framework (Nelson & Narens, 1990)
- Measures "retrospective confidence" — post-answer confidence ratings
- Uses Expected Calibration Error (ECE) as the primary metric
- Human baseline ECE is typically 0.10–0.20

Methodology:
1. Present diverse questions across domains and difficulty levels
2. Ask model to answer AND rate confidence (0–100)
3. Bin answers by confidence level
4. Compare stated confidence to actual accuracy per bin
5. Compute ECE = weighted average of |accuracy_bin - confidence_bin|

Score: Brier Skill Score (BSS = 1 - BS/BS_ref), which rewards both
calibration AND resolution (discrimination). Unlike 1-ECE, BSS properly
penalizes always-uncertain strategies and rewards models that assign high
confidence to correct answers and low confidence to incorrect ones.
BS_ref = climatological baseline (base_rate * (1 - base_rate)).

Shortcut Resistance:
- Questions span many domains (no single-domain memorisation helps)
- Mix of difficulty levels forces genuine uncertainty
- Confidence must be stated alongside the answer (no post-hoc adjustment)
"""

import kaggle_benchmarks as kbench
from dataclasses import dataclass
import numpy as np
import pandas as pd
import re
import json


# ─── Question Dataset ───────────────────────────────────────────────
# Inline all questions so the notebook is self-contained on Kaggle
# Source: data/calibration_questions.py (v2 handcrafted + procedural)

CALIBRATION_QUESTIONS = [
    # TIER 1: Easy
    {"question": "What is the chemical symbol for gold?", "answer": "Au", "domain": "chemistry", "difficulty": 1},
    {"question": "How many sides does a hexagon have?", "answer": "6", "domain": "math", "difficulty": 1},
    {"question": "What planet is known as the Red Planet?", "answer": "Mars", "domain": "astronomy", "difficulty": 1},
    {"question": "In which year did World War II end?", "answer": "1945", "domain": "history", "difficulty": 1},
    {"question": "What is the capital of Japan?", "answer": "Tokyo", "domain": "geography", "difficulty": 1},
    {"question": "What does DNA stand for?", "answer": "deoxyribonucleic acid", "domain": "biology", "difficulty": 1},
    {"question": "How many hearts does an octopus have?", "answer": "3", "domain": "biology", "difficulty": 1},
    {"question": "What is the smallest country in the world by area?", "answer": "Vatican City", "domain": "geography", "difficulty": 1},
    {"question": "In what year was Wikipedia launched?", "answer": "2001", "domain": "technology", "difficulty": 1},
    {"question": "What is the largest organ in the human body?", "answer": "skin", "domain": "biology", "difficulty": 1},
    {"question": "Who wrote the play 'Romeo and Juliet'?", "answer": "William Shakespeare", "domain": "literature", "difficulty": 1},
    {"question": "What is the boiling point of water in degrees Celsius at standard atmospheric pressure?", "answer": "100", "domain": "physics", "difficulty": 1},
    {"question": "How many bones are in the adult human body?", "answer": "206", "domain": "anatomy", "difficulty": 1},
    {"question": "What is the speed of sound in air at 20\u00b0C, in meters per second?", "answer": "343", "domain": "physics", "difficulty": 1},
    {"question": "Which country was the first to grant women the right to vote in national elections?", "answer": "New Zealand", "domain": "history", "difficulty": 1},
    # TIER 2: Medium
    {"question": "What is the half-life of Carbon-14, approximately in years?", "answer": "5730", "domain": "physics", "difficulty": 2},
    {"question": "In what year was the Treaty of Tordesillas signed, dividing the New World between Spain and Portugal?", "answer": "1494", "domain": "history", "difficulty": 2},
    {"question": "What is the densest naturally occurring element?", "answer": "osmium", "domain": "chemistry", "difficulty": 2},
    {"question": "How many time zones does Russia span?", "answer": "11", "domain": "geography", "difficulty": 2},
    {"question": "What element has the highest melting point?", "answer": "tungsten", "domain": "chemistry", "difficulty": 2},
    {"question": "In what year was the first network email sent by Ray Tomlinson?", "answer": "1971", "domain": "technology", "difficulty": 2},
    {"question": "How many US states border the Pacific Ocean?", "answer": "5", "domain": "geography", "difficulty": 2},
    {"question": "What is the Mohs hardness of quartz?", "answer": "7", "domain": "geology", "difficulty": 2},
    {"question": "In what year did the Berlin Wall fall?", "answer": "1989", "domain": "history", "difficulty": 2},
    {"question": "How many completed novels did Jane Austen write?", "answer": "6", "domain": "literature", "difficulty": 2},
    {"question": "In what year was the Battle of Hastings fought?", "answer": "1066", "domain": "history", "difficulty": 2},
    {"question": "What is the exact height of the Burj Khalifa in meters (to the tip)?", "answer": "828", "domain": "architecture", "difficulty": 2},
    {"question": "How many recognized countries are in Africa according to the United Nations?", "answer": "54", "domain": "geography", "difficulty": 2},
    {"question": "In what year was the first edition of the Encyclopaedia Britannica published?", "answer": "1768", "domain": "history", "difficulty": 2},
    {"question": "What is the standard atmospheric pressure at sea level in pascals?", "answer": "101325", "domain": "physics", "difficulty": 2},
    # TIER 3: Hard
    {"question": "What percentage of Earth's water is fresh water (not salt water)? Give to one decimal place.", "answer": "2.5", "domain": "earth science", "difficulty": 3},
    {"question": "What is the driest continent on Earth by average annual precipitation?", "answer": "Antarctica", "domain": "geography", "difficulty": 3},
    {"question": "Which country has the most islands in the world?", "answer": "Sweden", "domain": "geography", "difficulty": 3},
    {"question": "Is the Great Wall of China visible to the naked eye from low Earth orbit?", "answer": "No", "domain": "science", "difficulty": 3},
    {"question": "What is the value of the fine-structure constant (alpha) to 4 significant figures? Express as a decimal.", "answer": "0.007297", "domain": "physics", "difficulty": 3},
    {"question": "How many U.S. presidents have died while in office (including assassinations)?", "answer": "8", "domain": "history", "difficulty": 3},
    {"question": "Which planet in our solar system currently has the most known moons?", "answer": "Saturn", "domain": "astronomy", "difficulty": 3},
    {"question": "In what year was the earliest surviving photograph (by Nic\u00e9phore Ni\u00e9pce) taken?", "answer": "1826", "domain": "history", "difficulty": 3},
    {"question": "What is the exact value of the Planck constant h in J\u00b7s, as defined in the 2019 SI? Give all significant digits.", "answer": "6.62607015e-34", "domain": "physics", "difficulty": 3},
    {"question": "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost in dollars?", "answer": "0.05", "domain": "math", "difficulty": 3},
    {"question": "If you have a 4x4x4 cube made of 64 small unit cubes, and you paint the outside, how many unit cubes have exactly two painted faces?", "answer": "24", "domain": "math", "difficulty": 3},
    {"question": "In what year were human chromosomes correctly counted as 46 (not 48)?", "answer": "1955", "domain": "biology", "difficulty": 3},
    {"question": "What is the sum of all integers from 1 to 100?", "answer": "5050", "domain": "math", "difficulty": 3},
    {"question": "In the original Monty Hall problem, what is the probability of winning if you switch doors? Express as a fraction.", "answer": "2/3", "domain": "math", "difficulty": 3},
    {"question": "How many plays are in the traditional Shakespeare canon (First Folio plus Pericles)?", "answer": "37", "domain": "literature", "difficulty": 3},
    {"question": "What is the exact value of the Avogadro constant as defined in the 2019 SI redefinition, in mol\u207b\u00b9?", "answer": "6.02214076e23", "domain": "chemistry", "difficulty": 3},
    {"question": "In what year did Anders Celsius propose his temperature scale?", "answer": "1742", "domain": "history of science", "difficulty": 3},
    {"question": "What is the melting point of tungsten in degrees Celsius, rounded to the nearest degree?", "answer": "3422", "domain": "chemistry", "difficulty": 3},
    {"question": "How many edges does an icosahedron have?", "answer": "30", "domain": "math", "difficulty": 3},
    {"question": "What is the exact value of the Boltzmann constant k in J/K as defined in the 2019 SI?", "answer": "1.380649e-23", "domain": "physics", "difficulty": 3},
    # TIER 4: Very Hard
    {"question": "How many groups of order 8 exist up to isomorphism? (Counting all groups of order 8.)", "answer": "5", "domain": "abstract algebra", "difficulty": 4},
    {"question": "In a room of 23 people, what is the probability that at least two share a birthday? Give as a percentage rounded to the nearest whole number.", "answer": "50", "domain": "probability", "difficulty": 4},
    {"question": "What was the population of Liechtenstein in 2024, to the nearest thousand?", "answer": "40000", "domain": "geography", "difficulty": 4},
    {"question": "What is the only letter that does not appear in the name of any US state?", "answer": "Q", "domain": "trivia", "difficulty": 4},
    {"question": "How many two-digit prime numbers are there?", "answer": "21", "domain": "math", "difficulty": 4},
    {"question": "What is the surface area of a sphere with radius 7, in terms of exact value? Give a numerical answer rounded to 2 decimal places.", "answer": "615.75", "domain": "math", "difficulty": 4},
    {"question": "In a standard 52-card deck, what is the probability of being dealt a royal flush in 5-card poker? Express as '1 in N' where N is the answer.", "answer": "649740", "domain": "probability", "difficulty": 4},
    {"question": "Three people check into a hotel room that costs $30. They each pay $10. The manager realizes the room should only cost $25, so he gives $5 to the bellboy to return. The bellboy keeps $2 and gives back $1 to each guest. Now each guest has paid $9 (total $27), plus the bellboy has $2, totaling $29. Where is the missing dollar?", "answer": "There is no missing dollar. The $27 paid includes the $25 for the room plus the $2 the bellboy kept. The $29 figure incorrectly adds cost and tip.", "domain": "logic", "difficulty": 4},
    {"question": "What is the atomic number of the element Hassium?", "answer": "108", "domain": "chemistry", "difficulty": 4},
    {"question": "In what year was the Treaty of Nerchinsk signed between Russia and the Qing Dynasty?", "answer": "1689", "domain": "history", "difficulty": 4},
    {"question": "What is the 10th digit of pi after the decimal point?", "answer": "5", "domain": "math", "difficulty": 4},
    {"question": "How many perfect numbers are known to exist as of 2024?", "answer": "52", "domain": "math", "difficulty": 4},
    {"question": "If you fold a standard piece of paper (0.1mm thick) in half 42 times, approximately how thick would it be? Answer in kilometers to the nearest thousand.", "answer": "440000", "domain": "math", "difficulty": 4},
    {"question": "What is the name of the enzyme that catalyzes the conversion of carbon dioxide and water into glucose during the Calvin cycle in photosynthesis?", "answer": "RuBisCO", "domain": "biochemistry", "difficulty": 4},
    {"question": "If you have 12 identical-looking balls, one of which is either heavier or lighter than the rest, what is the minimum number of weighings on a balance scale needed to identify the odd ball and determine if it is heavier or lighter?", "answer": "3", "domain": "logic", "difficulty": 4},
    # TIER 5: Extreme
    {"question": "What is the exact year the Kingdom of Aksum (Axum) converted to Christianity under King Ezana?", "answer": "330", "domain": "history", "difficulty": 5},
    {"question": "What is the density of osmium in g/cm\u00b3, to 2 decimal places?", "answer": "22.59", "domain": "chemistry", "difficulty": 5},
    {"question": "In what year was the Oxford English Dictionary first fully published (all volumes of the first edition)?", "answer": "1928", "domain": "history", "difficulty": 5},
    {"question": "How many prime numbers are there between 1000 and 1100?", "answer": "16", "domain": "math", "difficulty": 5},
    {"question": "What is the exact area of Vatican City in square kilometers, to 2 decimal places?", "answer": "0.44", "domain": "geography", "difficulty": 5},
    {"question": "What specific article number of the UN Charter establishes the Security Council?", "answer": "23", "domain": "law", "difficulty": 5},
    {"question": "What is the speed of light in vacuum to 9 significant figures in m/s?", "answer": "299792458", "domain": "physics", "difficulty": 5},
    {"question": "In what year did Tjio and Levan publish their paper correctly establishing the human chromosome number as 46?", "answer": "1956", "domain": "biology", "difficulty": 5},
    {"question": "What is the sum of the reciprocals of all positive integers from 1 to 6, expressed as a fraction in lowest terms?", "answer": "49/20", "domain": "math", "difficulty": 5},
    {"question": "What was the exact date (day, month, year) of the Tunguska event?", "answer": "June 30, 1908", "domain": "history", "difficulty": 5},
    {"question": "How many known Mersenne primes exist as of 2024?", "answer": "52", "domain": "math", "difficulty": 5},
    {"question": "What is the shortest war in recorded history (between Britain and Zanzibar)? How many minutes did it last?", "answer": "38", "domain": "history", "difficulty": 5},
    {"question": "What specific year was the Antikythera mechanism estimated to have been built (the commonly cited date)?", "answer": "87 BC", "domain": "history", "difficulty": 5},
    {"question": "What is the 100th decimal digit of the mathematical constant e (Euler's number)?", "answer": "4", "domain": "math", "difficulty": 5},
    {"question": "What is the name of the Japanese era (neng\u014d) that began on May 1, 2019?", "answer": "Reiwa", "domain": "culture", "difficulty": 5},
    # === PROCEDURAL QUESTIONS (contamination-resistant) ===
    # TIER 1: Easy
    {"question": "What is 92 + 25?", "answer": "117", "domain": "arithmetic", "difficulty": 1},
    {"question": "What is 13 \u00d7 6?", "answer": "78", "domain": "arithmetic", "difficulty": 1},
    {"question": "How many meters are in 20 kilometers?", "answer": "20000", "domain": "conversion", "difficulty": 1},
    {"question": "How many hours are in 180 minutes?", "answer": "3", "domain": "conversion", "difficulty": 1},
    {"question": "What is 5 squared?", "answer": "25", "domain": "arithmetic", "difficulty": 1},
    {"question": "What is 24 + 97?", "answer": "121", "domain": "arithmetic", "difficulty": 1},
    {"question": "What is 46 \u00d7 3?", "answer": "138", "domain": "arithmetic", "difficulty": 1},
    {"question": "How many meters are in 50 kilometers?", "answer": "50000", "domain": "conversion", "difficulty": 1},
    {"question": "How many hours are in 120 minutes?", "answer": "2", "domain": "conversion", "difficulty": 1},
    {"question": "What is 3 squared?", "answer": "9", "domain": "arithmetic", "difficulty": 1},
    # TIER 2: Medium
    {"question": "Solve for x: 3x + 8 = -4", "answer": "-4", "domain": "algebra", "difficulty": 2},
    {"question": "What is the area of a triangle with base 21 cm and height 23 cm?", "answer": "241.5", "domain": "geometry", "difficulty": 2},
    {"question": "What is 25% of 63?", "answer": "15.75", "domain": "arithmetic", "difficulty": 2},
    {"question": "A car travels at 100 km/h for 5 hours. How far does it go (in km)?", "answer": "500", "domain": "physics", "difficulty": 2},
    {"question": "What is the remainder when 818 is divided by 19?", "answer": "1", "domain": "arithmetic", "difficulty": 2},
    {"question": "Solve for x: 8x + 15 = -9", "answer": "-3", "domain": "algebra", "difficulty": 2},
    {"question": "What is the area of a triangle with base 23 cm and height 12 cm?", "answer": "138", "domain": "geometry", "difficulty": 2},
    {"question": "What is 10% of 464?", "answer": "46.4", "domain": "arithmetic", "difficulty": 2},
    {"question": "A car travels at 40 km/h for 5 hours. How far does it go (in km)?", "answer": "200", "domain": "physics", "difficulty": 2},
    {"question": "What is the remainder when 532 is divided by 13?", "answer": "12", "domain": "arithmetic", "difficulty": 2},
    {"question": "Solve for x: 6x + 7 = -29", "answer": "-6", "domain": "algebra", "difficulty": 2},
    {"question": "What is the area of a triangle with base 29 cm and height 14 cm?", "answer": "203", "domain": "geometry", "difficulty": 2},
    {"question": "What is 15% of 102?", "answer": "15.3", "domain": "arithmetic", "difficulty": 2},
    {"question": "A car travels at 60 km/h for 1.5 hours. How far does it go (in km)?", "answer": "90", "domain": "physics", "difficulty": 2},
    {"question": "What is the remainder when 467 is divided by 13?", "answer": "12", "domain": "arithmetic", "difficulty": 2},
    # TIER 3: Hard
    {"question": "What is the sum of the first 8 terms of the arithmetic sequence starting at 10 with common difference 4?", "answer": "192", "domain": "math", "difficulty": 3},
    {"question": "How many ways can you choose 3 items from 10 distinct items (combinations)?", "answer": "120", "domain": "combinatorics", "difficulty": 3},
    {"question": "Find the roots of x\u00b2 + 1x - 20 = 0. Give the smaller root.", "answer": "-5", "domain": "algebra", "difficulty": 3},
    {"question": "What is the greatest common divisor (GCD) of 90 and 332?", "answer": "2", "domain": "math", "difficulty": 3},
    {"question": "An item costs $57. It is discounted by 20%, then 10% tax is added. What is the final price?", "answer": "50.16", "domain": "arithmetic", "difficulty": 3},
    {"question": "What is the sum of the first 9 terms of the arithmetic sequence starting at 4 with common difference 7?", "answer": "288", "domain": "math", "difficulty": 3},
    {"question": "How many ways can you choose 4 items from 5 distinct items (combinations)?", "answer": "5", "domain": "combinatorics", "difficulty": 3},
    {"question": "Find the roots of x\u00b2 x - 1 = 0. Give the smaller root.", "answer": "-1", "domain": "algebra", "difficulty": 3},
    {"question": "What is the greatest common divisor (GCD) of 90 and 487?", "answer": "1", "domain": "math", "difficulty": 3},
    {"question": "An item costs $49. It is discounted by 10%, then 8% tax is added. What is the final price?", "answer": "47.63", "domain": "arithmetic", "difficulty": 3},
    {"question": "What is the sum of the first 13 terms of the arithmetic sequence starting at 5 with common difference 5?", "answer": "455", "domain": "math", "difficulty": 3},
    {"question": "How many ways can you choose 3 items from 6 distinct items (combinations)?", "answer": "20", "domain": "combinatorics", "difficulty": 3},
    {"question": "Find the roots of x\u00b2 - 1x - 6 = 0. Give the smaller root.", "answer": "-2", "domain": "algebra", "difficulty": 3},
    {"question": "What is the greatest common divisor (GCD) of 393 and 186?", "answer": "3", "domain": "math", "difficulty": 3},
    {"question": "An item costs $29. It is discounted by 15%, then 10% tax is added. What is the final price?", "answer": "27.12", "domain": "arithmetic", "difficulty": 3},
]


# ─── Structured Output Schema ──────────────────────────────────────

@dataclass
class ConfidentAnswer:
    """Model's answer with a confidence rating."""
    answer: str           # The actual answer to the question
    confidence: int       # 0-100 confidence rating


# ─── Answer Verification ────────────────────────────────────────────

def normalize(text: str) -> str:
    """Normalize text for fuzzy matching."""
    text = text.lower().strip()
    # Remove articles, punctuation
    text = re.sub(r'\b(the|a|an)\b', '', text)
    text = re.sub(r'[^\w\s\.\+\-/]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def check_answer(model_answer: str, correct_answer: str) -> bool:
    """Check if model's answer matches the correct answer (fuzzy)."""
    model_norm = normalize(model_answer)
    correct_norm = normalize(correct_answer)

    # Direct containment (either direction)
    if correct_norm in model_norm or model_norm in correct_norm:
        return True

    # For numeric answers, try numeric comparison
    try:
        model_num = float(re.search(r'[\-\+]?[\d\.]+', model_norm).group())
        correct_num = float(re.search(r'[\-\+]?[\d\.]+', correct_norm).group())
        # Allow 5% tolerance for approximate numeric answers
        if correct_num == 0:
            return abs(model_num) < 0.01
        return abs(model_num - correct_num) / abs(correct_num) < 0.05
    except (ValueError, AttributeError, ZeroDivisionError):
        pass

    return False


# ─── Scoring Functions ──────────────────────────────────────────────

def compute_ece(confidences_0_100: list, accuracies: list, n_bins: int = 10) -> dict:
    """
    Compute Expected Calibration Error (diagnostic only — not used in final score).

    Returns dict with:
    - ece: float (0-1, lower = better calibrated)
    - bin_data: list of dicts with bin details
    - n_samples: int
    """
    confidences = np.array(confidences_0_100) / 100.0  # Normalize to 0-1
    accuracies = np.array(accuracies, dtype=float)

    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_data = []

    ece = 0.0
    total = len(confidences)

    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        bin_count = mask.sum()

        if bin_count == 0:
            bin_data.append({
                "bin": f"{lo:.1f}-{hi:.1f}",
                "count": 0,
                "avg_confidence": None,
                "avg_accuracy": None,
                "gap": None,
            })
            continue

        avg_conf = confidences[mask].mean()
        avg_acc = accuracies[mask].mean()
        gap = abs(avg_acc - avg_conf)
        ece += (bin_count / total) * gap

        bin_data.append({
            "bin": f"{lo:.1f}-{hi:.1f}",
            "count": int(bin_count),
            "avg_confidence": round(float(avg_conf), 3),
            "avg_accuracy": round(float(avg_acc), 3),
            "gap": round(float(gap), 3),
        })

    return {
        "ece": round(float(ece), 4),
        "bin_data": bin_data,
        "n_samples": total,
    }


# ─── The Benchmark Task ────────────────────────────────────────────

@kbench.task(name="Retrospective Calibration")
def metacog_calibration(llm) -> float:
    """
    Metacognitive Calibration Benchmark.

    Measures how well a model's stated confidence matches its actual accuracy.

    Score = max(0, BSS) where BSS = 1 - BS/BS_ref (Brier Skill Score).
    BSS rewards both calibration (confidence ≈ accuracy) and resolution
    (high confidence on correct, low on incorrect). An always-uncertain
    strategy scores ~0, not ~1 as with the old 1-ECE metric.

    Cognitive Science Basis: Nelson & Narens (1990) metamemory monitoring framework.
    Brier (1950) proper scoring rule; Murphy (1973) skill score decomposition.
    """
    confidences = []
    accuracies = []
    results_log = []

    for i, q in enumerate(CALIBRATION_QUESTIONS):
        with kbench.chats.new(f"q_{i}"):
            prompt = (
                f"Answer the following question. After your answer, rate your confidence "
                f"from 0 to 100 (where 0 = pure guess, 100 = absolutely certain).\n\n"
                f"Question: {q['question']}\n\n"
                f"Respond with ONLY a JSON object in this exact format:\n"
                f'{{"answer": "<your answer>", "confidence": <0-100>}}'
            )

            try:
                result = llm.prompt(prompt, schema=ConfidentAnswer)
                answer = result.answer
                confidence = max(0, min(100, result.confidence))
            except Exception:
                # Fallback: try to parse raw text
                raw = llm.prompt(prompt)
                try:
                    parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())
                    answer = str(parsed.get("answer", ""))
                    confidence = int(parsed.get("confidence", 50))
                    confidence = max(0, min(100, confidence))
                except Exception:
                    answer = raw
                    confidence = 50  # Default if parsing fails

            is_correct = check_answer(answer, q["answer"])
            confidences.append(confidence)
            accuracies.append(is_correct)

            results_log.append({
                "question": q["question"],
                "correct_answer": q["answer"],
                "model_answer": answer,
                "confidence": confidence,
                "is_correct": is_correct,
                "domain": q["domain"],
                "difficulty": q["difficulty"],
            })

    # Compute scoring metrics
    bss_raw = brier_skill_score(confidences, accuracies)
    score = round(max(0.0, bss_raw), 4)  # Clamp to [0, 1]

    # Proto3 omits zero-valued scalars from JSON (numericResult: {} instead of
    # numericResult: {value: 0}).  Use a tiny sentinel so the score survives
    # serialization and cache reloading.  1e-10 is below display precision.
    if score == 0.0:
        score = 1e-10

    # Diagnostic ECE (not used in final score)
    metrics = compute_ece(confidences, accuracies)

    # Log detailed results for analysis
    print(f"\n{'='*60}")
    print(f"METACOGNITIVE CALIBRATION RESULTS")
    print(f"{'='*60}")
    print(f"Questions answered: {metrics['n_samples']}")
    print(f"Overall accuracy: {sum(accuracies)/len(accuracies):.2%}")
    print(f"Mean confidence: {sum(confidences)/len(confidences):.1f}%")
    print(f"Brier Skill Score (raw): {bss_raw:.4f}")
    print(f"Score (clamped BSS): {score:.4f}")
    print(f"ECE (diagnostic): {metrics['ece']:.4f}")
    print(f"\nCalibration by bin:")
    for b in metrics["bin_data"]:
        if b["count"] > 0:
            print(f"  {b['bin']}: n={b['count']}, "
                  f"conf={b['avg_confidence']:.2f}, "
                  f"acc={b['avg_accuracy']:.2f}, "
                  f"gap={b['gap']:.3f}")

    # Log per-question details
    print(f"\nPer-question results:")
    for r in results_log:
        status = "✓" if r["is_correct"] else "✗"
        print(f"  {status} [{r['confidence']:3d}%] {r['question'][:50]}... "
              f"→ {r['model_answer'][:30]}")

    return score


# ─── Run ────────────────────────────────────────────────────────────
# On Kaggle: use kbench.llm
# Locally: this will error without the Kaggle proxy, but the code is testable


In [ ]:
metacog_calibration.run(kbench.llm)
